# Phase 4 — Regularization

**Team IDs:** 16002932, 16006616, 16004208
**Seed:** 42648

## Expectation

> We expect regularization to improve coefficient stability and remove redundant features introduced by the polynomial and interaction expansion. Because Phase 3 did not show severe random-split overfitting, we do not expect extremely large regularization strengths to improve validation R² dramatically. We expect L1 to produce a substantially smaller feature set, while Ridge may retain more correlated predictors. Elastic Net may provide a compromise between coefficient shrinkage and feature selection. We will select the final regularization strengths using the seeded validation set and then check whether the chosen model remains reasonable under the chronological setting.

In [1]:
import warnings, hashlib, json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

import gradient_descent as core
import polynomial_regression as p2
import bias_variance as p3
import regularization as p4

warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True)

SEED = 42648
DATA_DIR = Path('Datasets')
P3_STEM = 'bias_variance_artifacts'
P4_STEM = 'regularization_artifacts'
PLOTS = Path('plots'); PLOTS.mkdir(exist_ok=True)

## Load and Verify Phase 3 Artifacts

In [2]:
# Load Phase 3 Artifacts
P1_STEM = "gradient_descent_artifacts"
_, _, meta_p1, train_idx, val_idx = core.load_artifacts(P1_STEM)
prep_params = meta_p1

w_p3, b_p3, meta_p3 = p2.load_artifacts(P3_STEM)
P3_FEAT_NAMES = meta_p3["expanded_feature_names"]
poly_params = {
    "term_specs": meta_p3["term_specs"],
    "poly_means": meta_p3["poly_means"],
    "poly_stds": meta_p3["poly_stds"],
    "n_new_cols": meta_p3["n_new_cols"]
}

print("Phase 3 weights length:", len(w_p3))
print("Expanded feature names length:", len(P3_FEAT_NAMES))
assert len(w_p3) == len(P3_FEAT_NAMES)

print(f"Phase 3 SHA-256 Hash matches: {meta_p3['phase3_w_sha256'] == hashlib.sha256(w_p3.tobytes()).hexdigest()}")

train_all = pd.read_csv(DATA_DIR / 'train.csv')
train_df, val_df = train_test_split(train_all, test_size=0.20, random_state=SEED)
assert np.array_equal(train_df.index.values, train_idx)
assert np.array_equal(val_df.index.values, val_idx)
print("[PASS] Train/Validation indices match the seeded split.")

X_tr_raw, p1_names = core.transform(train_df, prep_params)
X_vl_raw, _        = core.transform(val_df, prep_params)

X_tr_exp, _ = p2.transform_poly(X_tr_raw, p1_names, poly_params)
X_vl_exp, _ = p2.transform_poly(X_vl_raw, p1_names, poly_params)

y_train = train_df['cnt'].values.astype(np.float64)
y_val   = val_df['cnt'].values.astype(np.float64)

# Test Phase 3 metrics
y_vl_pred_p3 = core.predict(X_vl_exp, w_p3, b_p3)
r2_p3 = core.r2_score_np(y_val, y_vl_pred_p3)
rmse_p3 = core.rmse_np(y_val, y_vl_pred_p3)
print(f"Phase 3 Val R²: {r2_p3:.4f}, RMSE: {rmse_p3:.4f}")


Phase 3 weights length: 589
Expanded feature names length: 589
Phase 3 SHA-256 Hash matches: True
[PASS] Train/Validation indices match the seeded split.
Phase 3 Val R²: 0.9104, RMSE: 53.0038


## Regularization Grid Search

- Ridge (L2)
- Lasso (L1)
- Elastic Net

In [3]:
# Common Protocol
MAX_ITER = 30000
TOL = 1e-9   # momentum (accelerated) fits are stopped on a tight relative-change tolerance
TOL_SELECT = 0.005   # parsimony tolerance on val R2 (see selection cell)
L_d = core.lipschitz_constant(X_tr_exp)
LR = p4.safe_lr(L_d, 0.0, 1.0)   # = 0.9/L_d, valid for Lasso (no smooth penalty)

# Edit 1: grid extended below the old edge. The old logspace(-3, 3) had its optimum on the
# lower edge; lambdas above ~10 leave < 20 features and add nothing.
lambdas = np.logspace(-4, 1, 26)[::-1]   # large to small for warm starts

print(f"L_d: {L_d:.6f}, LR (lasso): {LR:.6f}")
print(f"lambda grid: {lambdas[0]:.4g} ... {lambdas[-1]:.4g} ({len(lambdas)} values)")

L_d: 5.655143, LR (lasso): 0.159147
lambda grid: 10 ... 0.0001 (26 values)


In [4]:
# L2 (Ridge) Search -- Edit 2: step size 0.9/(L + lam) so the L2 curvature is respected
print("Starting Ridge Search...")
ridge_results = []
w_init = w_p3.copy()
b_init = float(b_p3)
n_diverged = 0

t0 = time.time()
for lam in lambdas:
    res = p4.fit_ridge(X_tr_exp, y_train, lr=p4.safe_lr(L_d, lam, 0.0), max_iter=MAX_ITER, lam=lam, tol=TOL,
                       w0=w_init, b0=b_init, X_val=X_vl_exp, y_val=y_val, momentum=True)
    if res['status'] == 'diverged':
        n_diverged += 1
        print(f"  lam={lam:.4g}: DIVERGED -- skipped, warm start kept")
        continue
    w_curr, b_curr = res['w'], res['b']

    y_tr_pred = core.predict(X_tr_exp, w_curr, b_curr)
    y_vl_pred = core.predict(X_vl_exp, w_curr, b_curr)

    ridge_results.append({
        'lam': lam,
        'val_r2': core.r2_score_np(y_val, y_vl_pred), 'train_r2': core.r2_score_np(y_train, y_tr_pred),
        'val_rmse': core.rmse_np(y_val, y_vl_pred), 'train_rmse': core.rmse_np(y_train, y_tr_pred),
        'nnz': int(np.sum(np.abs(w_curr) > 1e-8)), 'w': w_curr, 'b': b_curr,
        'status': res['status'], 'iters': res['n_iter']
    })
    w_init, b_init = w_curr, b_curr   # warm start (only from a healthy fit)

print(f"Ridge search completed in {time.time()-t0:.2f}s | fits: {len(ridge_results)} | diverged: {n_diverged}")
print("Statuses:", sorted({r['status'] for r in ridge_results}))

Starting Ridge Search...


Ridge search completed in 17.83s | fits: 26 | diverged: 0
Statuses: ['converged']


In [5]:
# L1 (Lasso) Search
print("Starting Lasso Search...")
lasso_results = []
w_init = w_p3.copy()
b_init = float(b_p3)
n_diverged = 0

t0 = time.time()
for lam in lambdas:
    res = p4.proximal_gd_fit(X_tr_exp, y_train, lr=p4.safe_lr(L_d, lam, 1.0), max_iter=MAX_ITER, lam=lam, alpha=1.0,
                             tol=TOL, w0=w_init, b0=b_init, X_val=X_vl_exp, y_val=y_val, momentum=True)
    if res['status'] == 'diverged':
        n_diverged += 1
        print(f"  lam={lam:.4g}: DIVERGED -- skipped, warm start kept")
        continue
    w_curr, b_curr = res['w'], res['b']

    y_tr_pred = core.predict(X_tr_exp, w_curr, b_curr)
    y_vl_pred = core.predict(X_vl_exp, w_curr, b_curr)

    lasso_results.append({
        'lam': lam,
        'val_r2': core.r2_score_np(y_val, y_vl_pred), 'train_r2': core.r2_score_np(y_train, y_tr_pred),
        'val_rmse': core.rmse_np(y_val, y_vl_pred), 'train_rmse': core.rmse_np(y_train, y_tr_pred),
        'nnz': int(np.sum(np.abs(w_curr) > 1e-8)), 'w': w_curr, 'b': b_curr,
        'status': res['status'], 'iters': res['n_iter']
    })
    w_init, b_init = w_curr, b_curr

print(f"Lasso search completed in {time.time()-t0:.2f}s | fits: {len(lasso_results)} | diverged: {n_diverged}")
print("Statuses:", sorted({r['status'] for r in lasso_results}))

Starting Lasso Search...


Lasso search completed in 46.18s | fits: 26 | diverged: 0
Statuses: ['converged']


In [6]:
# Elastic Net Search (alpha = L1 share / l1_ratio)
print("Starting Elastic Net Search...")
alphas = [0.25, 0.50, 0.90]                  # l1_ratio values searched
lambdas_en = np.logspace(-4, 1, 16)[::-1]    # coarser lambda grid than Ridge / Lasso
enet_results = []
n_diverged = 0

t0 = time.time()
for alpha in alphas:
    w_init = w_p3.copy()
    b_init = float(b_p3)
    for lam in lambdas_en:
        res = p4.proximal_gd_fit(X_tr_exp, y_train, lr=p4.safe_lr(L_d, lam, alpha), max_iter=MAX_ITER, lam=lam, alpha=alpha,
                                 tol=TOL, w0=w_init, b0=b_init, X_val=X_vl_exp, y_val=y_val, momentum=True)
        if res['status'] == 'diverged':
            n_diverged += 1
            print(f"  alpha={alpha} lam={lam:.4g}: DIVERGED -- skipped")
            continue
        w_curr, b_curr = res['w'], res['b']

        y_tr_pred = core.predict(X_tr_exp, w_curr, b_curr)
        y_vl_pred = core.predict(X_vl_exp, w_curr, b_curr)

        enet_results.append({
            'lam': lam, 'alpha': alpha,
            'val_r2': core.r2_score_np(y_val, y_vl_pred), 'train_r2': core.r2_score_np(y_train, y_tr_pred),
            'val_rmse': core.rmse_np(y_val, y_vl_pred), 'train_rmse': core.rmse_np(y_train, y_tr_pred),
            'nnz': int(np.sum(np.abs(w_curr) > 1e-8)), 'w': w_curr, 'b': b_curr,
            'status': res['status'], 'iters': res['n_iter']
        })
        w_init, b_init = w_curr, b_curr

print(f"Elastic Net search completed in {time.time()-t0:.2f}s | fits: {len(enet_results)} | diverged: {n_diverged}")
print("Statuses:", sorted({r['status'] for r in enet_results}))

Starting Elastic Net Search...


Elastic Net search completed in 39.70s | fits: 48 | diverged: 0
Statuses: ['converged']


## Model Selection & Plots

In [7]:
# Edit 3: parsimonious selection -- largest lambda within TOL_SELECT of the best val R2 (per method)
peak_ridge = max(ridge_results, key=lambda x: x['val_r2'])
peak_lasso = max(lasso_results, key=lambda x: x['val_r2'])
peak_enet  = max(enet_results,  key=lambda x: x['val_r2'])

sel_ridge = p4.select_parsimonious(ridge_results, TOL_SELECT)
sel_lasso = p4.select_parsimonious(lasso_results, TOL_SELECT)
sel_enet  = p4.select_parsimonious(enet_results,  TOL_SELECT)

print(f"Selection rule: largest lambda with val R2 >= (method best - {TOL_SELECT})\n")
print(f"{'method':<12}{'peak lam':>10}{'peak R2':>10} | {'chosen lam':>11}{'alpha':>7}{'val R2':>9}{'val RMSE':>10}{'nnz':>9}")
for name, pk, sl in [('Ridge', peak_ridge, sel_ridge), ('Lasso', peak_lasso, sel_lasso), ('ElasticNet', peak_enet, sel_enet)]:
    print(f"{name:<12}{pk['lam']:>10.4g}{pk['val_r2']:>10.4f} | {sl['lam']:>11.4g}{sl.get('alpha', float('nan')):>7.2f}"
          f"{sl['val_r2']:>9.4f}{sl['val_rmse']:>10.3f}{sl['nnz']:>9d}")

sel_models = [sel_ridge, sel_lasso, sel_enet]
print(f"\nPhase 3 (unregularised) val R2 for reference: {r2_p3:.4f}")
print(f"Lasso peak lambda {peak_lasso['lam']:.4g}; grid spans [{lambdas[-1]:.4g}, {lambdas[0]:.4g}]")

Selection rule: largest lambda with val R2 >= (method best - 0.005)

method        peak lam   peak R2 |  chosen lam  alpha   val R2  val RMSE      nnz
Ridge         0.000631    0.9112 |     0.01585    nan   0.9069    54.025      589
Lasso           0.0631    0.9125 |      0.2512    nan   0.9079    53.716      470
ElasticNet    0.004642    0.9114 |         0.1   0.90   0.9069    54.015      536

Phase 3 (unregularised) val R2 for reference: 0.9104
Lasso peak lambda 0.0631; grid spans [0.0001, 10]


In [8]:
# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ridge
lam_r = [x['lam'] for x in ridge_results]
vr2_r = [x['val_r2'] for x in ridge_results]
axes[0].plot(lam_r, vr2_r, 'b.-', label='Ridge (L2)')

# Lasso
lam_l = [x['lam'] for x in lasso_results]
vr2_l = [x['val_r2'] for x in lasso_results]
axes[0].plot(lam_l, vr2_l, 'r.-', label='Lasso (L1)')

# Elastic Net (show best alpha)
best_alpha = sel_enet['alpha']
enet_best_alpha = [x for x in enet_results if x['alpha'] == best_alpha]
lam_e = [x['lam'] for x in enet_best_alpha]
vr2_e = [x['val_r2'] for x in enet_best_alpha]
axes[0].plot(lam_e, vr2_e, 'g.-', label=f'E-Net (alpha={best_alpha})')

axes[0].set_xscale('log')
axes[0].set_title('Validation R² vs λ')
axes[0].set_xlabel('λ')
axes[0].set_ylabel('Validation R²')
axes[0].legend()

# Plot B: Val RMSE
vrmse_r = [x['val_rmse'] for x in ridge_results]
vrmse_l = [x['val_rmse'] for x in lasso_results]
vrmse_e = [x['val_rmse'] for x in enet_best_alpha]

axes[1].plot(lam_r, vrmse_r, 'b.-', label='Ridge')
axes[1].plot(lam_l, vrmse_l, 'r.-', label='Lasso')
axes[1].plot(lam_e, vrmse_e, 'g.-', label=f'E-Net')
axes[1].set_xscale('log')
axes[1].set_title('Validation RMSE vs λ')
axes[1].set_xlabel('λ')
axes[1].legend()

# Plot C: NNZ
nnz_r = [x['nnz'] for x in ridge_results]
nnz_l = [x['nnz'] for x in lasso_results]
nnz_e = [x['nnz'] for x in enet_best_alpha]

axes[2].plot(lam_r, nnz_r, 'b.-', label='Ridge')
axes[2].plot(lam_l, nnz_l, 'r.-', label='Lasso')
axes[2].plot(lam_e, nnz_e, 'g.-', label=f'E-Net')
axes[2].set_xscale('log')
axes[2].set_title('Non-Zero Features vs λ')
axes[2].set_xlabel('λ')
axes[2].legend()

for ax in axes:
    ax.axvline(sel_lasso['lam'], color='r', ls=':', alpha=0.5)
plt.tight_layout()
plt.savefig(PLOTS / '04_regularization_curves.png')
plt.close()

## Feature Ablation & Verdicts

In [9]:
# 1. Active features of the chosen Lasso
w_lasso = sel_lasso['w']
active_mask = np.abs(w_lasso) > 1e-8
active_features = [P3_FEAT_NAMES[i] for i in range(len(P3_FEAT_NAMES)) if active_mask[i]]
print(f"Lasso (lam={sel_lasso['lam']:.4g}) retained {len(active_features)} out of {len(P3_FEAT_NAMES)} features.")
print("Dropped:", [f for f in P3_FEAT_NAMES if f not in active_features])

# 2. Original-column -> derived-feature mapping
col_mapping = p4.get_original_col_mapping(P3_FEAT_NAMES)

# 3. Correlation partners between ORIGINAL columns (train set only)
_tr = train_df.copy()
_tr['dteday_n'] = (pd.to_datetime(_tr['dteday']) - pd.to_datetime(_tr['dteday']).min()).dt.days
partners = p4.partner_table(_tr, {
    'instant': 'instant', 'dteday': 'dteday_n', 'season': 'season', 'yr': 'yr', 'mnth': 'mnth', 'hr': 'hr',
    'holiday': 'holiday', 'weekday': 'weekday', 'workingday': 'workingday', 'weathersit': 'weathersit',
    'temp': 'temp', 'atemp': 'atemp', 'hum': 'hum', 'windspeed': 'windspeed'})

# 4. Drop-family ablation: remove all derived features of one original column, refit Lasso at chosen lambda
DELTA_THR, CORR_THR = 0.001, 0.85   # 0.85: yr (|r|=0.87 with instant) is a coarse function of the date
ablation_results = {}
for orig_col, derived_feats in col_mapping.items():
    partner, pcorr = partners[orig_col]
    if len(derived_feats) == 0:
        # instant is never a model input: its information enters only through days_since_start
        ablation_results[orig_col] = {'surviving_terms': 0, 'delta_r2': 0.0, 'delta_rmse': 0.0,
            'partner': partner, 'partner_corr': float(pcorr),
            'verdict': p4.column_verdict(0.0, partner, pcorr, DELTA_THR, CORR_THR)}
        continue
    mask = np.ones(X_tr_exp.shape[1], dtype=bool)
    for f in derived_feats:
        mask[P3_FEAT_NAMES.index(f)] = False

    res = p4.proximal_gd_fit(X_tr_exp[:, mask], y_train, lr=p4.safe_lr(L_d, sel_lasso['lam'], 1.0), max_iter=MAX_ITER,
                             lam=sel_lasso['lam'], alpha=1.0, tol=TOL, w0=w_lasso[mask], b0=sel_lasso['b'], momentum=True)
    y_vl_abl = core.predict(X_vl_exp[:, mask], res['w'], res['b'])
    abl_r2, abl_rmse = core.r2_score_np(y_val, y_vl_abl), core.rmse_np(y_val, y_vl_abl)
    delta_r2 = sel_lasso['val_r2'] - abl_r2
    surviving = [f for f in derived_feats if np.abs(w_lasso[P3_FEAT_NAMES.index(f)]) > 1e-8]
    ablation_results[orig_col] = {
        'surviving_terms': int(len(surviving)), 'delta_r2': float(delta_r2),
        'delta_rmse': float(abl_rmse - sel_lasso['val_rmse']),
        'partner': partner, 'partner_corr': float(pcorr),
        'verdict': p4.column_verdict(delta_r2, partner, pcorr, DELTA_THR, CORR_THR)}

print(f"\nVerdict rule: Useful if dR2 >= {DELTA_THR}; Redundant if not and |r| with another column >= {CORR_THR}; else Uninformative")
print(f"{'column':>11} {'n_feat':>6} {'kept':>5} {'dR2':>9} {'dRMSE':>8}  {'closest partner':<18} verdict")
for k, v in ablation_results.items():
    ptxt = v['partner'] + ' (' + format(v['partner_corr'], '.2f') + ')'
    print(f"{k:>11} {len(col_mapping[k]):>6} {v['surviving_terms']:>5} {v['delta_r2']:>9.4f} {v['delta_rmse']:>8.3f}  "
          f"{ptxt:<18} {v['verdict']}")

Lasso (lam=0.2512) retained 470 out of 589 features.
Dropped: ['hr_1', 'hr_2', 'hr_3', 'hr_4', 'hr_5', 'hr_6', 'hr_7', 'hr_8', 'hr_9', 'hr_17', 'hr_18', 'hr_19', 'hr_20', 'hr_21', 'hr_22', 'hr_23', 'season_2', 'season_3', 'mnth_2', 'mnth_3', 'mnth_4', 'mnth_5', 'mnth_6', 'mnth_7', 'mnth_8', 'mnth_9', 'mnth_10', 'mnth_11', 'mnth_12', 'weekday_2', 'weekday_3', 'weekday_4', 'weekday_5', 'weekday_6', 'holiday', 'yr', 'is_windspeed_zero', 'windspeed^3', 'hr_3*workingday', 'hr_16*workingday', 'hr_1*weather_2', 'hr_2*weather_2', 'hr_4*weather_2', 'hr_4*weather_3', 'hr_5*weather_2', 'hr_5*weather_3', 'hr_14*weather_2', 'hr_15*weather_2', 'hr_19*weather_2', 'hr_21*weather_2', 'hr_1*weekday_5', 'hr_2*weekday_3', 'hr_2*weekday_4', 'hr_2*weekday_5', 'hr_3*weekday_2', 'hr_3*weekday_3', 'hr_3*weekday_4', 'hr_3*weekday_5', 'hr_4*weekday_1', 'hr_4*weekday_2', 'hr_4*weekday_3', 'hr_4*weekday_4', 'hr_4*weekday_5', 'hr_5*weekday_1', 'hr_5*weekday_2', 'hr_5*weekday_3', 'hr_5*weekday_4', 'hr_5*weekday_5', 


Verdict rule: Useful if dR2 >= 0.001; Redundant if not and |r| with another column >= 0.85; else Uninformative
     column n_feat  kept       dR2    dRMSE  closest partner    verdict
    instant      0     0    0.0000    0.000  dteday (1.00)      Redundant (with dteday, |r|=1.00)
     dteday      1     1    0.0004    0.105  instant (1.00)     Redundant (with instant, |r|=1.00)
     season      3     1   -0.0000   -0.005  mnth (0.97)        Redundant (with mnth, |r|=0.97)
         yr     24    22    0.0306    8.277  instant (0.87)     Useful
       mnth    264   230    0.0113    3.208  season (0.97)      Useful
         hr    552   455    0.5752   90.905  hum (0.28)         Useful
    holiday      1     0   -0.0000   -0.000  workingday (0.25)  Uninformative
    weekday    144   100    0.0029    0.845  holiday (0.11)     Useful
 workingday     24    22    0.0190    5.275  holiday (0.25)     Useful
 weathersit     48    38    0.0103    2.938  hum (0.40)         Useful
       temp     26 

## Chronological Robustness Check

In [10]:
# Chronological split: same recipe as Phase 3 (sort by dteday, hr; first 80% train; preprocessor + poly refit on chrono-train)
train_all_chrono = train_all.sort_values(['dteday', 'hr']).reset_index(drop=True)
split_idx = int(len(train_all_chrono) * 0.8)
train_chrono = train_all_chrono.iloc[:split_idx]
val_chrono   = train_all_chrono.iloc[split_idx:]

pm_chr = core.fit_preprocessor(train_chrono)
X_tr_c, c_names = core.transform(train_chrono, pm_chr)
X_vl_c, _       = core.transform(val_chrono, pm_chr)
pp_chr = p3.fit_poly(X_tr_c, c_names, meta_p3['complexity_spec'])
X_tr_chr_exp, _ = p3.transform_poly(X_tr_c, c_names, pp_chr)
X_vl_chr_exp, _ = p3.transform_poly(X_vl_c, c_names, pp_chr)
y_tr_chr = train_chrono['cnt'].values.astype(np.float64)
y_vl_chr = val_chrono['cnt'].values.astype(np.float64)
L_chr = core.lipschitz_constant(X_tr_chr_exp)

# Both fits start from zeros: no weights learned on rows that sit in the chronological validation window
res_unreg = core.gd_fit(X_tr_chr_exp, y_tr_chr, lr=0.9 / L_chr, max_iter=100000, tol=1e-9, momentum=True)
r2_chr_unreg = core.r2_score_np(y_vl_chr, core.predict(X_vl_chr_exp, res_unreg['w'], res_unreg['b']))

res_chr = p4.proximal_gd_fit(X_tr_chr_exp, y_tr_chr, lr=p4.safe_lr(L_chr, sel_lasso['lam'], 1.0), max_iter=MAX_ITER,
                             lam=sel_lasso['lam'], alpha=1.0, tol=TOL, momentum=True)
y_vl_chr_pred = core.predict(X_vl_chr_exp, res_chr['w'], res_chr['b'])
chr_r2   = core.r2_score_np(y_vl_chr, y_vl_chr_pred)
chr_rmse = core.rmse_np(y_vl_chr, y_vl_chr_pred)

print(f"Chronological cut at {train_chrono['dteday'].max()} | train {len(train_chrono)} / val {len(val_chrono)}")
print(f"Unregularised GD (Phase 3 recipe): Val R2 {r2_chr_unreg:.4f}   (status {res_unreg['status']})")
print(f"Chosen Lasso (lam={sel_lasso['lam']:.4g})      : Val R2 {chr_r2:.4f}  RMSE {chr_rmse:.2f} (status {res_chr['status']})")
print(f"Seeded-split val R2 of the chosen Lasso: {sel_lasso['val_r2']:.4f}")

Chronological cut at 2012-08-05 | train 8708 / val 2178
Unregularised GD (Phase 3 recipe): Val R2 0.9063   (status converged)
Chosen Lasso (lam=0.2512)      : Val R2 0.8938  RMSE 70.91 (status converged)
Seeded-split val R2 of the chosen Lasso: 0.9079


## Save Artifacts

In [11]:
# Artifact Handoff
import json

final_model = sel_lasso

np.savez(P4_STEM + '.npz',
         l1_w=sel_lasso['w'], l1_b=np.array([sel_lasso['b']]),
         l2_w=sel_ridge['w'], l2_b=np.array([sel_ridge['b']]),
         en_w=sel_enet['w'],  en_b=np.array([sel_enet['b']]),
         train_idx=train_idx, val_idx=val_idx)

meta = {
    'phase3_w_sha256': meta_p3['phase3_w_sha256'],
    'phase4_l1_w_sha256': hashlib.sha256(sel_lasso['w'].tobytes()).hexdigest(),
    'phase4_l2_w_sha256': hashlib.sha256(sel_ridge['w'].tobytes()).hexdigest(),
    'phase4_en_w_sha256': hashlib.sha256(sel_enet['w'].tobytes()).hexdigest(),
    'feature_names': P3_FEAT_NAMES,
    
    'l1_lambda': sel_lasso['lam'],
    'ridge_lambda': sel_ridge['lam'],
    'elastic_net_lambda': sel_enet['lam'],
    'elastic_net_l1_ratio': sel_enet['alpha'],
    
    'l1_r2_train': sel_lasso['train_r2'], 'l1_r2_val': sel_lasso['val_r2'],
    'l1_rmse_train': sel_lasso['train_rmse'], 'l1_rmse_val': sel_lasso['val_rmse'],
    
    'ridge_r2_train': sel_ridge['train_r2'], 'ridge_r2_val': sel_ridge['val_r2'],
    'ridge_rmse_train': sel_ridge['train_rmse'], 'ridge_rmse_val': sel_ridge['val_rmse'],
    
    'en_r2_train': sel_enet['train_r2'], 'en_r2_val': sel_enet['val_r2'],
    'en_rmse_train': sel_enet['train_rmse'], 'en_rmse_val': sel_enet['val_rmse'],
    
    'selection_rule': f'largest lambda with val R2 within {TOL_SELECT} of method best',
    'selection_tolerance': TOL_SELECT,
    'lambda_grid': [float(x) for x in lambdas],
    'chrono_r2_lasso': float(chr_r2), 'chrono_r2_unregularised': float(r2_chr_unreg),
    'l1_nonzero_features': active_features,
    'original_column_verdicts': ablation_results
}

with open(P4_STEM + '.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f"Saved artifacts to {P4_STEM}.npz and .json")

Saved artifacts to regularization_artifacts.npz and .json


## Outcome

